# Homework: Quantum Circuits — Identities, Bell Measurements, Superdense Coding, and CHSH
**Lectures 3.5 & 3.6**

---

**Instructions**

- Complete every cell marked `### YOUR CODE HERE ###`.
- Do **not** rename variables that appear in the prompts — those names may be checked by the grader.
- Short-answer cells are graded on correctness and clarity, not length.
- Before submitting: `Kernel → Restart & Run All` and confirm zero errors.
- Upload this `.ipynb` file to Gradescope.

---

| Problem | Topic | Points |
|---------|-------|--------|
| 1 | CNOT circuit identities | 15 |
| 2 | Bell-state discrimination | 15 |
| 3 | Superdense coding | 20 |
| 4 | The CHSH game | 15 |
| **Total** | | **65** |


In [1]:
# Run this cell first
from qiskit import QuantumCircuit
from qiskit.quantum_info import Statevector, Operator
from qiskit_aer import AerSimulator
import numpy as np

simulator = AerSimulator()


---
## Problem 1 — CNOT Circuit Identities (20 pts)

Quantum circuits satisfy non-obvious algebraic identities — the same way Boolean logic has De Morgan's laws. Knowing these identities lets you simplify circuits and change between gate sets. In this problem you'll prove and verify two of the most important ones.

### Background

**Identity 1 — Basis change of CNOT.** Conjugating a CNOT by Hadamards on *both* qubits swaps its control and target:

$$
(H \otimes H) \cdot \text{CNOT}_{0 \to 1} \cdot (H \otimes H) = \text{CNOT}_{1 \to 0}
$$

where $\text{CNOT}_{0 \to 1}$ means qubit 0 is control, qubit 1 is target, and vice versa.

**Identity 2 — SWAP decomposition.** A SWAP gate can be built from three CNOTs:

$$
\text{SWAP} = \text{CNOT}_{0\to1} \cdot \text{CNOT}_{1\to0} \cdot \text{CNOT}_{0\to1}
$$

### 1a (8 pts) — Verify Identity 1 numerically

Build two circuits, each on 2 qubits, and extract their unitary matrices using `Operator`:
- `qc_lhs`: $H$ on both qubits, then $\text{CNOT}_{0\to 1}$, then $H$ on both qubits.
- `qc_rhs`: $\text{CNOT}_{1\to 0}$ (qubit 1 controls qubit 0).

Store the $4\times 4$ unitary matrices as `U_lhs` and `U_rhs`. Then verify they are equal by computing `np.allclose(U_lhs, U_rhs)` and storing the result in `identity1_holds` (should be `True`).

**Note on Qiskit ordering:** `cx(0, 1)` uses qubit 0 as control, qubit 1 as target. `cx(1, 0)` reverses this.

In [3]:
### SOLUTION ###

# LHS: H⊗H · CNOT_{0→1} · H⊗H
qc_lhs = QuantumCircuit(2)
qc_lhs.h(0)
qc_lhs.h(1)
qc_lhs.cx(0, 1)
qc_lhs.h(0)
qc_lhs.h(1)

# RHS: CNOT_{1→0}
qc_rhs = QuantumCircuit(2)
qc_rhs.cx(1, 0)

U_lhs = Operator(qc_lhs).data
U_rhs = Operator(qc_rhs).data

identity1_holds = np.allclose(U_lhs, U_rhs)

print("LHS unitary:")
print(np.round(np.real(U_lhs), 2))
print("RHS unitary:")
print(np.round(np.real(U_rhs), 2))
print(f"Identity 1 holds: {identity1_holds}")


LHS unitary:
[[1. 0. 0. 0.]
 [0. 1. 0. 0.]
 [0. 0. 0. 1.]
 [0. 0. 1. 0.]]
RHS unitary:
[[1. 0. 0. 0.]
 [0. 1. 0. 0.]
 [0. 0. 0. 1.]
 [0. 0. 1. 0.]]
Identity 1 holds: True


### 1b (4 pts) — Intuition for Identity 1

The Hadamard gate exchanges the $Z$-basis ($|0\rangle, |1\rangle$) and the $X$-basis ($|+\rangle, |−\rangle$). The CNOT is defined as: *if the control is $|1\rangle$ in the $Z$-basis, flip the target in the $Z$-basis*.

In 3–4 sentences, explain **conceptually** why conjugating by $H \otimes H$ swaps the roles of control and target. What does the CNOT look like when re-expressed in the $X$-basis?

**Your answer:**

Conjugating by $H \otimes H$ rewrites the circuit from the $Z$ basis into the $X$ basis because $H Z H = X$ and $H X H = Z$. A CNOT is “control in the $Z$ basis, flip in the $X$ sense,” so when both qubits are rotated by Hadamards, the role of “which qubit carries the condition” and “which qubit gets flipped” is exchanged. In that new basis, what used to be the target's bit flip becomes a phase condition, and what used to be the control's phase condition becomes the new bit flip. The net effect is exactly a CNOT with control and target swapped.


### 1c (8 pts) — Verify Identity 2 numerically and on a statevector

**Part i)** Build `qc_swap_3cnot` using three CNOTs as in Identity 2, and verify it equals a native SWAP by comparing `Operator` matrices. Store the result of `np.allclose(...)` in `identity2_holds`.

**Part ii)** Verify the identity concretely: start in state $|01\rangle$ (qubit 1 = $|0\rangle$, qubit 0 = $|1\rangle$, in Qiskit's ordering). Apply your three-CNOT SWAP and print the output statevector. It should equal $|10\rangle$.

In [ ]:
### SOLUTION ###

# Part i: Compare unitaries
qc_swap_3cnot = QuantumCircuit(2)
# CNOT_{0→1}, then CNOT_{1→0}, then CNOT_{0→1}
qc_swap_3cnot.cx(0, 1)
qc_swap_3cnot.cx(1, 0)
qc_swap_3cnot.cx(0, 1)

qc_swap_native = QuantumCircuit(2)
qc_swap_native.swap(0, 1)

identity2_holds = np.allclose(Operator(qc_swap_3cnot).data, Operator(qc_swap_native).data)
print(f"Identity 2 holds: {identity2_holds}")

# Part ii: Concrete statevector check
sv_01 = Statevector.from_label("01")  # Qiskit: qubit1=0, qubit0=1 → |01⟩
sv_after_swap = sv_01.evolve(qc_swap_3cnot)
print(f"Input:  {sv_01}")
print(f"Output: {sv_after_swap}  (should be |10⟩)")


Identity 2 holds: True

Input:  Statevector([0.+0.j, 1.+0.j, 0.+0.j, 0.+0.j], dims=(2, 2))
Output: Statevector([0.+0.j, 0.+0.j, 1.+0.j, 0.+0.j], dims=(2, 2))  (should be |10⟩)


---
## Problem 2 — Bell State Discrimination (20 pts)

In Lecture 3.5, the *reverse* Bell circuit (CNOT then H on qubit 0) maps each Bell state back to a computational-basis string. This is called a **Bell measurement** — it tells you which Bell state you have.

Lecture 3.5 noted: *"This reverse circuit is exactly what Bob does in superdense coding."* Before we get to superdense coding (Problem 3), let's build and test the Bell measurement itself.

Using **Qiskit's ordering** (`|q₁ q₀⟩`), the mapping for the Bell-state circuit (H on qubit 0, then `cx(0,1)`) followed by the reverse Bell circuit is:

<table>
  <tr><th>Bell state</th><th>Measurement outcome after reverse Bell circuit</th></tr>
  <tr><td>|Φ⁺⟩ = (|00⟩ + |11⟩)/√2</td><td><code>00</code></td></tr>
  <tr><td>|Ψ⁺⟩ = (|01⟩ + |10⟩)/√2</td><td><code>10</code></td></tr>
  <tr><td>|Φ⁻⟩ = (|00⟩ − |11⟩)/√2</td><td><code>01</code></td></tr>
  <tr><td>|Ψ⁻⟩ = (|01⟩ − |10⟩)/√2</td><td><code>11</code></td></tr>
</table>


### 2a (10 pts) — Build a Bell discriminator

Write a function `bell_discriminator(bell_label)` that:
1. Prepares the named Bell state (`'Phi+'`, `'Phi-'`, `'Psi+'`, or `'Psi-'`) **from `|00⟩` using circuit gates** — do **not** hardcode a statevector.
2. Appends the Bell measurement (CNOT, then `H` on qubit 0).
3. Measures both qubits.
4. Returns the `QuantumCircuit`.

Using Qiskit's ordering, the Bell-state circuit (`H` on qubit 0, then `cx(0, 1)`) maps computational-basis inputs as follows:

| Input to Bell-state circuit | Output Bell state |
|---|---|
| `00` | `Phi+` |
| `01` | `Phi-` |
| `10` | `Psi+` |
| `11` | `Psi-` up to global phase |

So the cleanest way to prepare each Bell state is:
- first prepare the appropriate computational-basis input with **`X` gates only**,
- then apply the Bell-state circuit (`H` on qubit 0, then `cx(0, 1)`).

Run each circuit for 1024 shots and store the four counts dicts in a list `discrimination_results`. Each should be deterministic (one outcome only).

*Hint:* in Qiskit, a basis label like `'10'` means qubit 1 is `1` and qubit 0 is `0`.


In [ ]:
def bell_discriminator(bell_label):
    """
    Prepare the given Bell state, then apply Bell measurement.
    bell_label: 'Phi+', 'Phi-', 'Psi+', or 'Psi-'
    Returns: QuantumCircuit with measurement
    """
    qc = QuantumCircuit(2, 2)

    # Step 1: prepare the computational-basis input that generates the desired Bell state
    mapping = {
        'Phi+': '00',
        'Phi-': '01',
        'Psi+': '10',
        'Psi-': '11',
    }
    input_label = mapping[bell_label]

    # If input_label[1] == '1', flip qubit 0.
    # If input_label[0] == '1', flip qubit 1.
    if input_label[1] == '1':
        qc.x(0)
    if input_label[0] == '1':
        qc.x(1)

    # Step 2: Bell-state circuit
    qc.h(0)
    qc.cx(0, 1)

    # Step 3: Bell measurement (reverse Bell circuit)
    qc.cx(0, 1)
    qc.h(0)

    qc.measure([0, 1], [0, 1])
    return qc


bell_labels   = ['Phi+', 'Psi+', 'Phi-', 'Psi-']
expected_bits = ['00',   '10',   '01',   '11']

discrimination_results = []
print(f"{'Bell state':<10}  {'Expected':<10}  {'Measured (top outcome)'}")
print("-" * 50)

for label, expected in zip(bell_labels, expected_bits):
    qc = bell_discriminator(label)
    counts = simulator.run(qc, shots=1024).result().get_counts()
    discrimination_results.append(counts)
    top = max(counts, key=counts.get)
    print(f"|{label:<9}  {expected:<10}  {top}  {'✓' if top == expected else '✗'}")


Bell state  Expected    Measured (top outcome)
--------------------------------------------------
|Phi+       00          00  ✓
|Psi+       10          10  ✓
|Phi-       01          01  ✓
|Psi-       11          11  ✓


### 2b (6 pts) — A superposition of Bell states

Prepare the state

$$
\frac{1}{\sqrt{2}}\bigl(|\Phi^+\rangle + |\Phi^-\rangle\bigr)
$$

directly as a statevector, then apply the **reverse Bell circuit only** (no measurement yet) and print the resulting statevector.

Store the output in `sv_superposition_out`.

Then answer:
1. Which computational-basis states appear after the reverse Bell circuit?
2. What probabilities do you get if you measure at that point?
3. Why is this a good reminder that the Bell-measurement circuit is just a **unitary basis change** before the final measurement, not some nonlinear “Bell-label detector”?


In [ ]:
### SOLUTION ###

# Build (|Phi+> + |Phi->)/sqrt(2) from amplitudes.
# In Qiskit's computational-basis ordering [|00>, |01>, |10>, |11>]:
phi_plus  = np.array([1/np.sqrt(2), 0, 0,  1/np.sqrt(2)], dtype=complex)
phi_minus = np.array([1/np.sqrt(2), 0, 0, -1/np.sqrt(2)], dtype=complex)

superposition_vec = (phi_plus + phi_minus) / np.linalg.norm(phi_plus + phi_minus)
superposition = Statevector(superposition_vec)

qc_reverse = QuantumCircuit(2)
# Reverse Bell circuit: CNOT, then H on qubit 0
qc_reverse.cx(0, 1)
qc_reverse.h(0)

sv_superposition_out = superposition.evolve(qc_reverse)
print("Output statevector:", sv_superposition_out)
print("Probabilities:", sv_superposition_out.probabilities_dict())


Output statevector: Statevector([0.70710678+0.j, 0.70710678+0.j, 0.        +0.j,
             0.        +0.j], dims=(2, 2))
Probabilities: {'00': 0.4999999999999999, '01': 0.4999999999999999}


### 2c (4 pts) — Short answer

Without doing any new simulation, predict what the reverse Bell circuit does to

$$
\frac{1}{\sqrt{2}}\bigl(|\Phi^+\rangle - |\Phi^-\rangle\bigr).
$$

Use linearity and the Bell-to-computational-basis mapping from 2a. You may give your answer either
- as a superposition of computational-basis states, or
- in product-state form.

Be explicit about which qubit is in superposition in Qiskit’s ordering.


**Your answer:**

By linearity,

( |Φ⁺⟩ − |Φ⁻⟩ )/√2  →  ( |00⟩ − |01⟩ )/√2.

That can be written as |0⟩₁ ⊗ |−⟩₀, where the subscript reminds you of Qiskit's ordering |q₁ q₀⟩. So qubit 1 is definitely in |0⟩, while qubit 0 is the qubit left in superposition.


---
## Problem 3 — Superdense Coding (25 pts)

Lecture 3.5 noted: *"Running the circuit backward — first CNOT, then Hadamard — takes Bell states back to the computational basis. This is how you measure in the Bell basis ... This reverse circuit is exactly what Bob does in superdense coding."*

This problem builds the full superdense coding protocol.

### Protocol overview

Alice and Bob share |Φ⁺⟩. Alice holds qubit 0, Bob holds qubit 1. Alice wants to send a 2-bit message (m₁, m₀) ∈ {0,1}² by applying a gate to **her qubit only**, then sending it to Bob. Bob then does a Bell measurement on both qubits.

With **Qiskit's ordering** and Alice on qubit 0, the clean encoding is:

<table>
  <tr><th>Message (m₁, m₀)</th><th>Gate Alice applies to qubit 0</th><th>Resulting Bell state</th></tr>
  <tr><td><code>00</code></td><td>I (identity)</td><td>|Φ⁺⟩</td></tr>
  <tr><td><code>01</code></td><td>Z</td><td>|Φ⁻⟩</td></tr>
  <tr><td><code>10</code></td><td>X</td><td>|Ψ⁺⟩</td></tr>
  <tr><td><code>11</code></td><td>XZ (or ZX, up to global phase)</td><td>|Ψ⁻⟩ up to global phase</td></tr>
</table>

So in code, m₁ is naturally encoded by applying X to qubit 0, and m₀ is naturally encoded by applying Z to qubit 0. Bob's decoding is the reverse Bell circuit: apply `cx(0, 1)` and then `h(0)`.


### 3a (10 pts) — Build the encoding and verify the Bell states

Complete the function `alice_encode(message)` below. It should:
1. Take a 2-character message string interpreted as `m1m0` (`'00'`, `'01'`, `'10'`, `'11'`).
2. Start from $|\Phi^+\rangle$.
3. Apply Alice's encoding gate to **qubit 0 only**.
4. Return the output **statevector** (not the circuit).

Print the output statevector for all four messages. Verify they match the four Bell states.

In [ ]:
def alice_encode(message):
    """
    Encode a 2-bit message using superdense coding.
    message: '00', '01', '10', or '11' (m1 m0)
    Returns: Statevector of the encoded two-qubit state.
    """
    # Prepare Phi+
    qc = QuantumCircuit(2)
    qc.h(0)
    qc.cx(0, 1)
    
    # Apply Alice's gate to qubit 0 based on the message
    m1, m0 = int(message[0]), int(message[1])
    # m1 is encoded with X, m0 with Z
    if m1 == 1:
        qc.x(0)
    if m0 == 1:
        qc.z(0)
    
    return Statevector.from_instruction(qc)


expected_states = {
    '00': '|Φ+⟩ = (|00⟩ + |11⟩)/√2',
    '01': '|Φ-⟩ = (|00⟩ - |11⟩)/√2',
    '10': '|Ψ+⟩ = (|01⟩ + |10⟩)/√2',
    '11': '|Ψ-⟩ up to global phase = (|01⟩ - |10⟩)/√2',
}

for msg in ['00', '01', '10', '11']:
    sv = alice_encode(msg)
    print(f"Message {msg} → {sv}")
    print(f"  (Expected: {expected_states[msg]})
")


Message 00 → Statevector([0.70710678+0.j, 0.        +0.j, 0.        +0.j,
             0.70710678+0.j], dims=(2, 2))
  (Expected: |Φ+⟩ = (|00⟩ + |11⟩)/√2)

Message 01 → Statevector([ 0.70710678+0.j,  0.        +0.j,  0.        +0.j,
            -0.70710678+0.j], dims=(2, 2))
  (Expected: |Φ-⟩ = (|00⟩ - |11⟩)/√2)

Message 10 → Statevector([0.        +0.j, 0.70710678+0.j, 0.70710678+0.j,
             0.        +0.j], dims=(2, 2))
  (Expected: |Ψ+⟩ = (|01⟩ + |10⟩)/√2)

Message 11 → Statevector([ 0.        +0.j, -0.70710678+0.j,  0.70710678+0.j,
             0.        +0.j], dims=(2, 2))
  (Expected: |Ψ-⟩ up to global phase = (|01⟩ - |10⟩)/√2)


### 3b (10 pts) — Build Bob's decoder and run the full protocol

Complete the function `superdense_protocol(message)` that:
1. Starts from $|00\rangle$.
2. Creates $|\Phi^+\rangle$ (shared entanglement).
3. Applies Alice's encoding (from 3a logic, but now in-circuit, not statevector).
4. Applies Bob's decoding: CNOT with qubit 0 as control and qubit 1 as target, then $H$ on qubit 0.
5. Measures both qubits into 2 classical bits.
6. Returns the circuit.

Run each of the four messages for 1024 shots. Store the list of counts dicts in `superdense_results`. The decoded measurement should deterministically return the original message.

*Grading note:* `superdense_results[i]` should correspond to messages `['00', '01', '10', '11'][i]`.

In [ ]:
def superdense_protocol(message):
    """
    Full superdense coding circuit.
    message: '00', '01', '10', '11'
    Returns: QuantumCircuit with measurement
    """
    qc = QuantumCircuit(2, 2)
    
    # Step 1 & 2: Shared Bell state
    qc.h(0)
    qc.cx(0, 1)
    
    qc.barrier(label="shared entanglement")
    
    # Step 3: Alice encodes
    m1, m0 = int(message[0]), int(message[1])
    if m1 == 1:
        qc.x(0)
    if m0 == 1:
        qc.z(0)
    
    qc.barrier(label="Alice encodes")
    
    # Step 4: Bob decodes (reverse Bell circuit)
    qc.cx(0, 1)
    qc.h(0)
    
    qc.barrier(label="Bob decodes")
    
    # Step 5: Measure
    qc.measure([0, 1], [0, 1])
    
    return qc


messages = ['00', '01', '10', '11']
superdense_results = []

print(f"{'Message sent':<15} {'Decoded (top outcome)':<25} {'Correct?'}")
print("-" * 50)

for msg in messages:
    qc = superdense_protocol(msg)
    counts = simulator.run(qc, shots=1024).result().get_counts()
    superdense_results.append(counts)
    top = max(counts, key=counts.get)
    # Qiskit ordering: outcome string is q1q0, so m1=outcome[0], m0=outcome[1]
    print(f"{msg:<15} {top:<25} {'✓' if top == msg else '✗'}")


Message sent    Decoded (top outcome)     Correct?
--------------------------------------------------
00              00                        ✓
01              01                        ✓
10              10                        ✓
11              11                        ✓


### 3c (5 pts) — Draw the circuit for message `'11'`

Draw the full circuit for message `'11'` and confirm visually that Alice's operations (X then Z on qubit 0) appear between the two barriers.

In [ ]:
### SOLUTION ###
print(superdense_protocol('11').draw(output='text'))


     ┌───┐     shared entanglement ┌───┐┌───┐ Alice encodes      ┌───┐ Bob decodes      ┌─┐   
q_0: ┤ H ├──■──────────────░────────┤ X ├┤ Z ├──────░───────────■──┤ H ├─────░───────────┤M├───
     └───┘┌─┴─┐            ░        └───┘└───┘      ░         ┌─┴─┐└───┘     ░           └╥┘┌─┐
q_1: ─────┤ X ├────────────░────────────────────────░─────────┤ X ├──────────░────────────╫─┤M├
          └───┘            ░                        ░         └───┘          ░            ║ └╥┘
 c: 2/══════════════════════════════════════════════════════════════════════════════════════╩══╩═
                                                                                           0  1 


---
## Problem 4 — The CHSH Game (15 pts)

Recall from Lecture 3.6 the CHSH game:
- The referee sends Alice a bit $x$ and Bob a bit $y$.
- They reply with bits $a$ and $b$.
- **Win condition:** $a \oplus b = x \wedge y$.

The quantum strategy uses shared $|\Phi^+\rangle$ and measurement rotations:
- Alice: rotate by $\alpha = 0$ if $x=0$, and $\alpha = \pi/4$ if $x=1$.
- Bob: rotate by $\beta = \pi/8$ if $y=0$, and $\beta = -\pi/8$ if $y=1$.
- Implement each basis choice with `qc.ry(-2 * angle, qubit)`.

Your goal is to compare the best simple classical strategy from the lecture (`a=b=0` always) with the quantum strategy.


### 4a (10 pts) — Simulate the CHSH win rate

Complete the function `chsh_win_rate(strategy, shots=8000)` below. The `strategy` argument is either `'quantum'` or `'classical'`.

- **Quantum:** for each $(x, y)$, build a circuit using $|\Phi^+\rangle$ and the rotation angles above, measure, and tally wins.
- **Classical:** Alice and Bob always answer $a=0$, $b=0$.

The function should return the overall win rate as a float between 0 and 1.

*Hint:* after measurement, Qiskit returns the bitstring in the order `q1 q0`, so use
- `a = int(outcome[1])` for Alice/qubit 0,
- `b = int(outcome[0])` for Bob/qubit 1.


In [ ]:
def chsh_win_rate(strategy, shots=8000):
    """
    Simulate the CHSH game.
    strategy: 'quantum' or 'classical'
    Returns: overall win rate as a float
    """
    total_wins = 0
    total_games = 0

    for x in [0, 1]:
        for y in [0, 1]:

            if strategy == 'classical':
                # Always answer a=0, b=0
                a, b = 0, 0
                win = (a ^ b) == (x & y)
                total_wins += shots if win else 0
                total_games += shots

            elif strategy == 'quantum':
                # Build a 2-qubit circuit:
                # 1. Create |Phi+>
                # 2. Rotate qubit 0 (Alice) by alpha(x)
                # 3. Rotate qubit 1 (Bob) by beta(y)
                # 4. Measure both qubits

                qc = QuantumCircuit(2, 2)
                qc.h(0)
                qc.cx(0, 1)

                alpha = 0 if x == 0 else np.pi/4
                beta  = np.pi/8 if y == 0 else -np.pi/8

                qc.ry(-2 * alpha, 0)
                qc.ry(-2 * beta, 1)
                qc.measure([0, 1], [0, 1])

                counts = simulator.run(qc, shots=shots).result().get_counts()
                for outcome, count in counts.items():
                    a = int(outcome[1])  # qubit 0 = rightmost
                    b = int(outcome[0])  # qubit 1 = leftmost
                    if (a ^ b) == (x & y):
                        total_wins += count
                total_games += shots

    return total_wins / total_games


classical_rate = chsh_win_rate('classical')
quantum_rate   = chsh_win_rate('quantum')

print(f"Classical win rate : {classical_rate:.3f}  (theoretical max: 0.750)")
print(f"Quantum win rate   : {quantum_rate:.3f}  (Tsirelson bound:  {np.cos(np.pi/8)**2:.3f})")


Classical win rate : 0.750  (theoretical max: 0.750)
Quantum win rate   : 0.854  (Tsirelson bound:  0.854)


### 4b (5 pts) — Short answer

Answer both parts in 4–6 sentences total.

1. Your quantum win rate should come out around 85%, above the classical 75% limit. What **resource** is responsible for that advantage, and why do the angle choices matter?
2. A hypothetical no-signaling theory can, in principle, win CHSH with probability 100% (for example, a PR-box model). Why does that **not** contradict no-signaling, and what bound from quantum mechanics would such a theory violate instead?


**Your answer here:**

The quantum advantage comes from **shared entanglement**: Alice and Bob start with |Φ⁺⟩, so their outcomes can show stronger correlations than any classical shared-randomness strategy. The angle choices matter because they pick measurement bases that align with the Bell-state correlations in exactly the way that maximizes the CHSH winning probability. A PR-box-type model can reach 100% CHSH success without signaling because no-signaling only says one party's marginal distribution cannot depend on the other party's input; it does **not** forbid stronger-than-quantum nonlocal correlations. What such a model would violate is the **Tsirelson bound** from quantum mechanics, equivalently the quantum CHSH bound S ≤ 2√2 or the win-probability bound cos²(π/8).


---